In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torchvision.transforms as transforms

# Test one image from folders

In [2]:
test_image = "../data/pnemonia_images/00000061_015.png"

img = Image.open(test_image)
img.mode

'L'

In [ ]:
transform = transforms.ToTensor()
# Convert PIL image to tensor
tensor = transform(img)

torch.Size([1, 128, 128])

# Check patient duplicates

In [6]:
# download_path
download_path = "/mnt/c/Users/User/Documents/0Unicamp/IA376N/Projeto/archive"

# Loads the dataset with metadata
file_path_in_dataset = "Data_Entry_2017.csv"

# Construct the full path to the CSV file within the downloaded dataset
full_csv_path = os.path.join(download_path, file_path_in_dataset)

# Now, load the CSV using pandas.read_csv with the correct encoding and error handling.
df = pd.read_csv(
    full_csv_path
)

df.drop('Unnamed: 11', axis=1, inplace=True)
df

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143
...,...,...,...,...,...,...,...,...,...,...,...
112115,00030801_001.png,Mass|Pneumonia,1,30801,39,M,PA,2048,2500,0.168,0.168
112116,00030802_000.png,No Finding,0,30802,29,M,PA,2048,2500,0.168,0.168
112117,00030803_000.png,No Finding,0,30803,42,F,PA,2048,2500,0.168,0.168
112118,00030804_000.png,No Finding,0,30804,30,F,PA,2048,2500,0.168,0.168


In [ ]:
# download_path
download_path = "/mnt/c/Users/User/Documents/0Unicamp/IA376N/Projeto/archive"

# Loads the dataset with metadata
file_path_in_dataset = "Data_Entry_2017.csv"

# Construct the full path to the CSV file within the downloaded dataset
full_csv_path = os.path.join(download_path, file_path_in_dataset)

# Now, load the CSV using pandas.read_csv with the correct encoding and error handling.
df = pd.read_csv(
    full_csv_path
)

df.drop('Unnamed: 11', axis=1, inplace=True)
df

healthy_images = os.listdir("../data/healthy_images")
pnemonia_images = os.listdir("../data/pnemonia_images")

len(df.loc[df["Image Index"].isin(pnemonia_images)]["Patient ID"].unique())
len(df.loc[df["Image Index"].isin(healthy_images)]["Patient ID"].unique())

In [14]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# CONFIG
# =========================================================

SEED = 42

TRAIN_RATIO = 0.80
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# NIH metadata path
download_path = "/mnt/c/Users/User/Documents/0Unicamp/IA376N/Projeto/archive"

# CSV
csv_path = os.path.join(
    download_path,
    "Data_Entry_2017.csv"
)

# Original image folders
healthy_dir = "../data/healthy_images"
pneumonia_dir = "../data/pnemonia_images"

# Output split folder
output_dir = "../data/processed"


# =========================================================
# LOAD METADATA
# =========================================================

df = pd.read_csv(csv_path)

df.drop("Unnamed: 11", axis=1, inplace=True)

healthy_images = os.listdir(healthy_dir)
pneumonia_images = os.listdir(pneumonia_dir)

# Keep only images that physically exist
healthy_df = df[
    df["Image Index"].isin(healthy_images)
].copy()

pneumonia_df = df[
    df["Image Index"].isin(pneumonia_images)
].copy()

healthy_df["domain"] = "healthy"
pneumonia_df["domain"] = "pneumonia"


# =========================================================
# FILTER HEALTHY PATIENTS
# =========================================================
# Keep only healthy patients with <= 2 images
# This avoids patients dominating the dataset
# =========================================================

healthy_patient_counts = (
    healthy_df.groupby("Patient ID")
    .size()
)

valid_healthy_patients = healthy_patient_counts[
    healthy_patient_counts <= 2
].index

healthy_df = healthy_df[
    healthy_df["Patient ID"].isin(valid_healthy_patients)
]

print("=" * 50)
print("HEALTHY PATIENT FILTER")
print("=" * 50)

print(f"Healthy images after filter: {len(healthy_df)}")
print(f"Healthy patients after filter: {healthy_df['Patient ID'].nunique()}")


# =========================================================
# BALANCE DATASET
# =========================================================
# healthy patients = 2x pneumonia patients
# =========================================================

pneumonia_patients = pneumonia_df["Patient ID"].unique()

num_pneumonia_patients = len(pneumonia_patients)

healthy_patients = healthy_df["Patient ID"].unique()

np.random.seed(SEED)

selected_healthy_patients = np.random.choice(
    healthy_patients,
    size=min(
        len(healthy_patients),
        num_pneumonia_patients * 2
    ),
    replace=False
)

healthy_df = healthy_df[
    healthy_df["Patient ID"].isin(selected_healthy_patients)
]

print("\n" + "=" * 50)
print("BALANCED DATASET")
print("=" * 50)

print(f"Pneumonia patients: {num_pneumonia_patients}")
print(f"Healthy patients: {len(selected_healthy_patients)}")


# =========================================================
# SPLIT FUNCTION
# =========================================================

def split_patients(patient_ids):

    patient_ids = np.array(patient_ids)

    np.random.shuffle(patient_ids)

    total = len(patient_ids)

    train_end = int(total * TRAIN_RATIO)

    val_end = train_end + int(total * VAL_RATIO)

    train_patients = patient_ids[:train_end]

    val_patients = patient_ids[train_end:val_end]

    test_patients = patient_ids[val_end:]

    return {
        "train": train_patients,
        "val": val_patients,
        "test": test_patients
    }


# =========================================================
# SPLIT EACH DOMAIN SEPARATELY
# =========================================================

healthy_splits = split_patients(
    selected_healthy_patients
)

pneumonia_splits = split_patients(
    pneumonia_patients
)


# =========================================================
# COPY FILES
# =========================================================

def copy_split_files(df, splits, domain, source_dir):

    source_dir = Path(source_dir)

    for split_name, patient_ids in splits.items():

        split_df = df[
            df["Patient ID"].isin(patient_ids)
        ]

        target_dir = (
            Path(output_dir)
            / split_name
            / domain
        )

        target_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        for image_name in split_df["Image Index"]:

            source_path = source_dir / image_name

            target_path = target_dir / image_name

            if source_path.exists():

                shutil.copy(
                    source_path,
                    target_path
                )

        print("\n" + "-" * 40)
        print(f"{domain.upper()} - {split_name.upper()}")
        print("-" * 40)

        print(
            f"Patients: "
            f"{split_df['Patient ID'].nunique()}"
        )

        print(
            f"Images: "
            f"{len(split_df)}"
        )


# =========================================================
# COPY HEALTHY
# =========================================================

copy_split_files(
    healthy_df,
    healthy_splits,
    domain="healthy",
    source_dir=healthy_dir
)

# =========================================================
# COPY PNEUMONIA
# =========================================================

copy_split_files(
    pneumonia_df,
    pneumonia_splits,
    domain="pneumonia",
    source_dir=pneumonia_dir
)


# =========================================================
# FINAL CHECK
# =========================================================

print("\n" + "=" * 50)
print("FINAL OVERLAP CHECK")
print("=" * 50)

healthy_train = set(healthy_splits["train"])
healthy_val = set(healthy_splits["val"])
healthy_test = set(healthy_splits["test"])

pne_train = set(pneumonia_splits["train"])
pne_val = set(pneumonia_splits["val"])
pne_test = set(pneumonia_splits["test"])

print("\nHealthy overlap:")

print(
    "Train ∩ Val:",
    len(healthy_train & healthy_val)
)

print(
    "Train ∩ Test:",
    len(healthy_train & healthy_test)
)

print(
    "Val ∩ Test:",
    len(healthy_val & healthy_test)
)

print("\nPneumonia overlap:")

print(
    "Train ∩ Val:",
    len(pne_train & pne_val)
)

print(
    "Train ∩ Test:",
    len(pne_train & pne_test)
)

print(
    "Val ∩ Test:",
    len(pne_val & pne_test)
)

print("\nDone.")

HEALTHY PATIENT FILTER
Healthy images after filter: 22634
Healthy patients after filter: 18990

BALANCED DATASET
Pneumonia patients: 286
Healthy patients: 572

----------------------------------------
HEALTHY - TRAIN
----------------------------------------
Patients: 457
Images: 537

----------------------------------------
HEALTHY - VAL
----------------------------------------
Patients: 57
Images: 74

----------------------------------------
HEALTHY - TEST
----------------------------------------
Patients: 58
Images: 71

----------------------------------------
PNEUMONIA - TRAIN
----------------------------------------
Patients: 228
Images: 260

----------------------------------------
PNEUMONIA - VAL
----------------------------------------
Patients: 28
Images: 29

----------------------------------------
PNEUMONIA - TEST
----------------------------------------
Patients: 30
Images: 33

FINAL OVERLAP CHECK

Healthy overlap:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

Pneumonia over